<a href="https://colab.research.google.com/github/eteitelbaum/code-satp/blob/main/model-data-size-experiments/action_type/exp_actiontype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()  # Apply threshold
#     labels = labels.astype(int)

#     hamming = hamming_loss(labels, predictions)
#     subset_acc = accuracy_score(labels, predictions)
#     report = classification_report(labels, predictions, output_dict=True, zero_division=0)

#     return {
#         "hamming_loss": hamming,
#         "subset_accuracy": subset_acc,
#         "precision_micro": report["micro avg"]["precision"],
#         "recall_micro": report["micro avg"]["recall"],
#         "f1_micro": report["micro avg"]["f1-score"],
#     }

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# =======================
# Generalized Dataset Class
# =======================
class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),
        }

# =======================
# Function to Compute Metrics
# =======================


from sklearn.metrics import classification_report, hamming_loss, accuracy_score

def compute_metricss(eval_pred):
    """
    Compute evaluation metrics for multi-label classification.
    Includes Hamming Loss, Subset Accuracy, and Classification Report for all labels.
    """
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()  # Apply threshold
    labels = labels.astype(int)

    # Hamming Loss
    hamming = hamming_loss(labels, predictions)

    # Subset Accuracy
    subset_acc = accuracy_score(labels, predictions)

    # Classification Report
    report = classification_report(
        labels, predictions, target_names=data.columns[1:], zero_division=0, output_dict=True
    )

    # Print complete report for reference
    print("\nFull Classification Report:")
    print(classification_report(labels, predictions, target_names=data.columns[:-1], zero_division=0))

    # Summary Metrics for Trainer
    metrics = {
        "hamming_loss": hamming,
        "subset_accuracy": subset_acc,
    }
    metrics.update(report)
    return metrics


# =============================
# Reusable Training Function
# =============================

def train_transformer_model(model_name, data, max_len=512, test_size=0.1, val_size=0.1, batch_size=40, epochs=3):
    """
    Generalized function to train a transformer model for multi-label classification.
    Args:
        model_name: Name of the pre-trained model (e.g., "bert-base-uncased", "distilbert-base-uncased").
        data: Pandas DataFrame with columns "incident_summary" and multi-label columns.
        max_len: Maximum sequence length.
        batch_size: Batch size for training and evaluation.
        epochs: Number of training epochs.
    """
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=data.shape[1] - 1,  # Number of labels (all columns except "incident_summary")
        problem_type="multi_label_classification",
    )
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    # Split data into train, val, and test
    X = data["incident_summary"]
    y = data.drop('incident_summary', axis=1).values

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=test_size, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=val_size, random_state=42)

    # Create datasets
    train_dataset = MultiLabelDataset(X_train.tolist(), y_train, tokenizer, max_len)
    val_dataset = MultiLabelDataset(X_val.tolist(), y_val, tokenizer, max_len)
    test_dataset = MultiLabelDataset(X_test.tolist(), y_test, tokenizer, max_len)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir="./results",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model='eval_hamming_loss',
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metricss,
    )

    # Train and Evaluate
    trainer.train()

    # Final Evaluation on Test Set
    test_results = trainer.evaluate(test_dataset)
    print("Test Set Results:", test_results)

    return trainer, test_results


In [ ]:
fraction_labels = {
    1/32: "3%",   # 1/32 = 3.125%
    1/16: "6%",   # 1/16 = 6.25%
    1/8:  "12%",  # 1/8  = 12.5%
    1/4:  "25%",
    1/2:  "50%",
    1.0:  "100%"
}

model_name_labels = {
    "bert-base-cased": "BERT Base (cased)",
    "snowood1/ConfliBERT-scr-cased": "ConfliBERT-scr-cased",
    "FacebookAI/roberta-base": "RoBERTa Base",
    "distilbert-base-cased": "DistilBERT (cased)",
    "xlnet-base-cased": "XLNet Base (cased)",
    "google/electra-base-discriminator": "ELECTRA Base"
}

fractions = [1/32, 1/16, 1/8, 1/4, 1/2, 1.0]

models_list = [
    "bert-base-cased",
    "snowood1/ConfliBERT-scr-cased",
    "FacebookAI/roberta-base",
    "distilbert-base-cased",
    "xlnet-base-cased",
    "google/electra-base-discriminator"
]

In [ ]:
def run_all_experiments_and_save(df_full, output_csv="experiment_results.csv"):
    """
    1. Iterates over the defined fractions & model list
    2. Samples df_full according to fraction
    3. Trains & evaluates using train_multiclass_model_3way_split
    4. Saves the collected results in a DataFrame
    5. Exports to CSV

    Args:
        df_full (pd.DataFrame): Full dataset with columns [label_col, text_col].
        output_csv (str): File path to save the experiment results.
    Returns:
        results_df (pd.DataFrame): Contains experiment results for analysis.
    """
    results_list = []

    for frac in fractions:
        # Sample a fraction of the data
        subset_size = int(len(df_full) * frac)
        df_subset = df_full.sample(n=subset_size, random_state=42)

        # Friendly fraction label if you want
        frac_label = fraction_labels.get(frac, f"{frac*100:.1f}%")
        print(f"\n=== DATA FRACTION: {frac} ({subset_size} rows) ===")

        for model_name in models_list:
            # Model label
            model_label = model_name_labels.get(model_name, model_name)
            print(f"Training model: {model_label}")

            # Train & evaluate
            # write the model funtion here
            trainer, test_results = train_transformer_model(model_name, df_subset, max_len=512, test_size=0.1, val_size=0.1, batch_size=16, epochs=2)


            # Build a result dict
            run_result = {
                "fraction_raw": frac,
                "fraction_label": frac_label,
                "subset_size": subset_size,
                "model_raw": model_name,
                "model_label": model_label
            }

            # Flatten the nested dictionary
            for key, value in test_results.items():
                if isinstance(value, dict):
                    for subkey, subvalue in value.items():
                        # Create new key names like "armed_assault_precision"
                        run_result[f"{key}_{subkey}"] = subvalue
                else:
                    run_result[key] = value

            # Append to results_list
            results_list.append(run_result)

    # Convert to DataFrame
    results_df = pd.DataFrame(results_list)
    # Save to CSV
    results_df.to_csv(output_csv, index=False)
    print(f"\nResults saved to {output_csv}")

    # also save to JSON
    # results_df.to_json("experiment_results.json", orient="records")

    return results_df




#**Read from GitHub**

In [ ]:
import pandas as pd

# Corrected URL to access the raw CSV data
url = 'https://raw.githubusercontent.com/eteitelbaum/code-satp/main/model-data-size-experiments/action_type/action_type.csv'

try:
    data = pd.read_csv(url)
    print(data.head())
except Exception as e:
    print(f"Error loading CSV from URL: {e}")

  perpetrator                                   incident_summary
0    Security  An alleged arms supplier to the Communist Part...
1      Maoist  A Kamareddy dalam (squad) member belonging to ...
2    Security  Senior CPI-Maoist 'Polit Bureau' and 'central ...
3      Maoist  A TDP leader and former Sarpanch of Jerrela Gr...
4      Maoist  The CPI-Maoist cadres blasted coffee pulping u...


# **Read From Drive**

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv('/content/drive/MyDrive/SATP_data/action_type.csv')


# **Loop Exe**

In [ ]:

#
# Example usage:
final_results_df = run_all_experiments_and_save(data, output_csv="experiment_results.csv")

# Now you can inspect final_results_df in Python:
print(final_results_df.head())

# If you want to do more analysis, you can pivot, group, or plot the data.



=== DATA FRACTION: 0.03125 (310 rows) ===
Training model: BERT Base (cased)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval

Epoch,Training Loss,Validation Loss,Hamming Loss,Subset Accuracy,Arrest,Bombing,Infrastructure,Surrender,Seizure,Abduction,Incident Summary,Micro avg,Macro avg,Weighted avg,Samples avg,Runtime,Samples Per Second,Steps Per Second
1,0.631600,0.491015,0.169312,0.000000,"{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 4.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 4.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}",0.817800,33.015000,2.446000
2,0.458300,0.452601,0.169312,0.000000,"{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 4.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 4.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 32.0}",0.875400,30.842000,2.285000



Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.00      0.00      0.00         7
        arrest       0.00      0.00      0.00         8
       bombing       0.00      0.00      0.00         3
infrastructure       0.00      0.00      0.00         5
     surrender       0.00      0.00      0.00         4
       seizure       0.00      0.00      0.00         4
     abduction       0.00      0.00      0.00         1

     micro avg       0.00      0.00      0.00        32
     macro avg       0.00      0.00      0.00        32
  weighted avg       0.00      0.00      0.00        32
   samples avg       0.00      0.00      0.00        32


Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.00      0.00      0.00         7
        arrest       0.00      0.00      0.00         8
       bombing       0.00      0.00      0.00         3
infrastructure       0.00      0.00      0


Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.00      0.00      0.00         1
        arrest       0.00      0.00      0.00         0
       bombing       0.00      0.00      0.00         1
infrastructure       0.00      0.00      0.00         1
     surrender       0.00      0.00      0.00         1
       seizure       0.00      0.00      0.00         0
     abduction       0.00      0.00      0.00         1

     micro avg       0.00      0.00      0.00         5
     macro avg       0.00      0.00      0.00         5
  weighted avg       0.00      0.00      0.00         5
   samples avg       0.00      0.00      0.00         5

Test Set Results: {'eval_loss': 0.5474851131439209, 'eval_hamming_loss': 0.17857142857142858, 'eval_subset_accuracy': 0.0, 'eval_arrest': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0}, 'eval_bombing': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0}, 'eval_

...


Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.94      0.94      0.94       161
        arrest       0.98      0.97      0.97       138
       bombing       0.94      0.94      0.94        51
infrastructure       0.97      0.74      0.84        43
     surrender       0.98      1.00      0.99        44
       seizure       0.95      0.87      0.90       104
     abduction       0.94      0.73      0.82        22

     micro avg       0.96      0.91      0.94       563
     macro avg       0.96      0.88      0.92       563
  weighted avg       0.96      0.91      0.93       563
   samples avg       0.96      0.94      0.94       563


Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.97      0.96      0.96       161
        arrest       0.99      0.99      0.99       138
       bombing       0.88      0.98      0.93        51
infrastructure       0.89      0.74      0


Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.81      0.93      0.87        14
        arrest       1.00      1.00      1.00        15
       bombing       1.00      0.90      0.95        10
infrastructure       1.00      0.60      0.75         5
     surrender       1.00      1.00      1.00         5
       seizure       0.92      0.92      0.92        13
     abduction       1.00      0.75      0.86         4

     micro avg       0.94      0.91      0.92        66
     macro avg       0.96      0.87      0.91        66
  weighted avg       0.95      0.91      0.92        66
   samples avg       0.96      0.94      0.94        66

Test Set Results: {'eval_loss': 0.0860959067940712, 'eval_hamming_loss': 0.02857142857142857, 'eval_subset_accuracy': 0.8, 'eval_arrest': {'precision': 0.8125, 'recall': 0.9285714285714286, 'f1-score': 0.8666666666666667, 'support': 14.0}, 'eval_bombing': {'precision': 1.0, 'recall': 1.0, 'f1-sc

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-base-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Hamming Loss,Subset Accuracy,Arrest,Bombing,Infrastructure,Surrender,Seizure,Abduction,Incident Summary,Micro avg,Macro avg,Weighted avg,Samples avg,Runtime,Samples Per Second,Steps Per Second
1,0.200600,0.172232,0.036835,0.807175,"{'precision': 0.8994082840236687, 'recall': 0.9440993788819876, 'f1-score': 0.9212121212121213, 'support': 161.0}","{'precision': 0.9924242424242424, 'recall': 0.9492753623188406, 'f1-score': 0.9703703703703703, 'support': 138.0}","{'precision': 0.9523809523809523, 'recall': 0.7843137254901961, 'f1-score': 0.8602150537634409, 'support': 51.0}","{'precision': 0.9629629629629629, 'recall': 0.6046511627906976, 'f1-score': 0.7428571428571429, 'support': 43.0}","{'precision': 1.0, 'recall': 0.9545454545454546, 'f1-score': 0.9767441860465116, 'support': 44.0}","{'precision': 0.90625, 'recall': 0.8365384615384616, 'f1-score': 0.87, 'support': 104.0}","{'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 22.0}","{'precision': 0.9409448818897638, 'recall': 0.8490230905861457, 'f1-score': 0.892623716153128, 'support': 563.0}","{'precision': 0.8162037773988323, 'recall': 0.7247747922236627, 'f1-score': 0.7630569820356553, 'support': 563.0}","{'precision': 0.9058403466451014, 'recall': 0.8490230905861457, 'f1-score': 0.8729961486627029, 'support': 563.0}","{'precision': 0.9159192825112108, 'recall': 0.8800448430493274, 'f1-score': 0.8878176382660689, 'support': 563.0}",13.709300,32.533000,2.042000



Full Classification Report:
                precision    recall  f1-score   support

 armed_assault       0.90      0.94      0.92       161
        arrest       0.99      0.95      0.97       138
       bombing       0.95      0.78      0.86        51
infrastructure       0.96      0.60      0.74        43
     surrender       1.00      0.95      0.98        44
       seizure       0.91      0.84      0.87       104
     abduction       0.00      0.00      0.00        22

     micro avg       0.94      0.85      0.89       563
     macro avg       0.82      0.72      0.76       563
  weighted avg       0.91      0.85      0.87       563
   samples avg       0.92      0.88      0.89       563



In [ ]:
import os


def save_to_drive(df, filename, folder_name="actiontype"):
    base_path = "/content/drive/MyDrive/SATP_data/paper/exp"
    drive_path = os.path.join(base_path, folder_name)

    # Create the folder if it doesn't exist
    if not os.path.exists(drive_path):
        os.makedirs(drive_path)

    filepath = os.path.join(drive_path, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ DataFrame saved to: {filepath}")

save_to_drive(final_results_df, "exp_actiontype_results.csv")
